# TCR-pMHC transport in an immunological synapse

Rebuild of Isaacson & Zhang, *Bull. Math. Biol.* **87**:14 (2024), section IV.D, with hard-sphere
volume exclusion. Their scheme carries no two-body interactions, and that section states that it
therefore ignores size-based segregation forces. This study puts that exclusion in and asks what
it changes.

**This notebook drives; it does not reimplement.** Every run and every figure comes from the
modules beside it, so a result produced here and a result produced from the shell are the same
code path:

| module | role |
|---|---|
| `synapse_model.py` | geometry, species, equivalent diameters, gauge, rates |
| `synapse_run.py` | builds and runs one arm, reduces frames to radial histograms and 2D maps |
| `synapse_rate_gate.py` | absolute gate of the bimolecular rate against well-mixed kinetics |
| `synapse_analyze.py` | inspection, seed aggregation, the radial and Boltzmann figures |
| `synapse_maps_fig.py` | the two-row contour panels |
| `lmrun.sh`, `lmrun_bg.sh` | container wrappers (foreground, detached) |

Simulations need the GPU and the `lm2.6` container, so they are launched through the wrappers
rather than executed in this kernel. Analysis cells run wherever numpy and matplotlib are
available.

## 1. Where the data is

`out/` is a symlink into the campaign tree at `/raid/racda/campaigns/tcr_pmhc`, so the `.npz`
files are not in the git repository. The cell below reports what is present.

In [ ]:
import os, glob, json
import numpy as np

OUT = 'out'
print('out -> %s\n' % os.path.realpath(OUT))
rows = []
for p in sorted(glob.glob(os.path.join(OUT, '*.npz'))):
    try:
        d = np.load(p, allow_pickle=True)
        m = json.loads(str(d['meta']))
        rows.append((os.path.basename(p), m.get('arm'), m.get('gauge'), m.get('f0'),
                     m.get('tsim'), 'maps' in d, m.get('no_reaction')))
    except Exception as e:
        rows.append((os.path.basename(p), 'unreadable: %s' % e, '', '', '', '', ''))
print('%-26s %-9s %-8s %6s %7s %6s %s' % ('file', 'arm', 'gauge', 'f0', 'tsim', 'maps', 'no_rxn'))
for r in rows:
    print('%-26s %-9s %-8s %6s %7s %6s %s' % r)

## 2. The model, and the one non-obvious parameter

The molecules are anchored to a membrane, so their crowding is areal, but the excess free energy
is a three-dimensional functional reading a volume fraction out of a cubic voxel. Left alone it
spreads a jammed monolayer through 40 nm of $z$ and reports a dilute gas. The diameter handed to
the functional is therefore the sphere carrying the same second virial coefficient **per unit
area** of the layer. That changes a parameter, never a functional.

In [ ]:
import synapse_model as M

a_core, a_ann = M.core_annulus_areas_um2()
print('voxel %.0f nm | disk R %.1f um | core R %.1f um | pSMAC R %.1f um'
      % (M.H_NM, M.R_DISK_UM, M.R_CORE_UM, M.R_PSMAC_UM))
print('core %.2f um^2, seeded annulus %.2f um^2, ratio %.2f\n' % (a_core, a_ann, a_ann / a_core))
print('%-10s %6s %6s %8s %10s %10s' % ('species', 'span', 'width', 'D', 'equiv', 'lattice'))
for name, span, width, D, _c in M.SPECIES:
    eq = M.kihara_equiv_diameter(span, width)
    print('%-10s %6.1f %6.1f %8.2f %10.2f %10.2f'
          % (name, span, width, D, eq, M.slab_equivalent_diameter(eq)))
print('\ngauge weights (multiplier on f0):')
for g in M.GAUGES:
    print('  %-8s %s' % (g, '  '.join('%s=%.2g' % (n, M.coupling_weight(n, g))
                                      for n, _, _, _, _ in M.SPECIES)))

## 3. Gate the rate before trusting anything

Run this first if the engine build has moved. It seeds the whole disk uniformly with the field
off and compares the product count against the integrated well-mixed kinetics. Two failure modes
it has already caught: `sim.reaction(..., regions=None)` silently assigns the reaction to no
region so it never fires, and adaptive tau makes the chemistry run about $1/\tau_{\rm scale}$
times too fast per unit reported clock.

In [ ]:
print("""./lmrun.sh 0 python synapse_run.py --arm point --r-core 0 --f0 0 \\
      --tsim 0.2 --frames 4 --density 1410 --out out/rate_gate.npz
./lmrun.sh 0 python synapse_rate_gate.py out/rate_gate.npz""")
print('\nlast recorded pass: median ratio 0.999 at production geometry and density')

## 4. The runs

Canonical configuration: gauge `db`, so free TCR and pMHC feel no active force and only the
complex couples to the ramp, exactly as Isaacson & Zhang specify. 40 nm voxels, 288 x 288 x 8,
one active layer, 200 s, adaptive tau off.

The zero-force arm is the control: with `--f0 0` nothing drives the complexes inward, so it
isolates what the ramp is responsible for. Note that in the `db` gauge the reaction is also no
longer suppressed toward the centre, so this arm forms more complexes overall, not the same
population redistributed.

In [ ]:
MT = '0,2,10,100,200'
for f0, tag in ((1, 'M_react'), (0, 'F0_react')):
    for gpu, arm in ((0, 'point'), (1, 'excluded')):
        print('./lmrun_bg.sh %d %s_%s python synapse_run.py --arm %s --gauge db --f0 %d \\\n'
              '      --tsim 200 --frames 100 --density 1410 --map-times "%s" --out out/%s_%s.npz'
              % (gpu, tag, arm, arm, f0, MT, tag, arm))

## 5. Contour panels

The two-row analogue of their Figure 8. Point particles above, volume excluded below, one shared
colour axis so the rows are comparable.

Two presentation decisions. Density is plotted per unit area, not per subvolume, because their
mesh polygon is 7093 nm² against our 1600 nm² voxel and raw counts are not comparable between the
two papers. And the maps are block-summed to 80 nm cells for display, within ten percent of their
polygon: a 40 nm voxel holds about two complexes, so voxel-resolved counts are mostly shot noise.

The colour ramp is **not defined here**. It belongs to the Kip3 microtubule-traffic study
(`../lmrddme_validation/kip3_traffic/kip3_palette.py`) and this figure imports it, so the two
applications in the paper share one scale instead of drifting apart.

In [ ]:
for tag, what in (('M_react', 'with the centripetal ramp, f0 = 1 kT/um'),
                  ('F0_react', 'no centripetal force, f0 = 0')):
    print('# %s' % what)
    print('./lmrun.sh 5 python synapse_maps_fig.py \\\n'
          '      --point out/%s_point.npz --excluded out/%s_excluded.npz \\\n'
          '      --times "0,2,10,100,200" --width 6.5 --out out/fig_maps_%s\n'
          % (tag, tag, tag.lower().replace('_react', '')))

In [ ]:
from IPython.display import Image, display
for stem, cap in (('fig_maps_react', 'with the ramp'), ('fig_maps_f0', 'no force')):
    p = os.path.join(OUT, stem + '.png')
    if os.path.exists(p):
        print(cap)
        display(Image(filename=p))
    else:
        print('%s not built yet' % p)

## 5a. The model as a cartoon

`tcr_scheme_fig.py` draws the schematic: the contact edge-on with the binding reaction, the
contact zone from above with the drive, and the potential the complex feels. It follows the
layout of Dharan & Farago's Fig. 1 and takes their geometry, but it is deliberately not their
model, and the docstring lists the four differences: the binding is explicit here and absent
there, only the product is driven, there is no LFA1-ICAM1 and no pinning obstacles, and the
membranes are flat because this model has no membrane elasticity to make them undulate.

In [ ]:
print('./lmrun.sh 5 python tcr_scheme_fig.py --out out/fig_scheme')

p = os.path.join(OUT, 'fig_scheme.png')
display(Image(filename=p)) if os.path.exists(p) else print('%s not built yet' % p)

## 6. Numbers

Seed-aggregated, three seeds each, of the canonical configuration.

In [ ]:
import importlib, synapse_analyze as A
importlib.reload(A)

sets = {
    'transport, point':    ['T200_db_point', 'R1_T200_db_point', 'R2_T200_db_point'],
    'transport, excluded': ['T200_db_excluded', 'R1_T200_db_excluded', 'R2_T200_db_excluded'],
    'reactive, point':     ['L200_db_point', 'R1_L200_db_point', 'R2_L200_db_point'],
    'reactive, excluded':  ['L200_db_excluded', 'R1_L200_db_excluded', 'R2_L200_db_excluded'],
}
for label, stems in sets.items():
    paths = [os.path.join(OUT, s + '.npz') for s in stems]
    if all(os.path.exists(p) for p in paths):
        A.aggregate(paths, label)
        print()

## 7. What the runs said

**Transport only, against an exact reference.** Complexes pre-seeded, no reaction, so the
stationary profile is exactly Boltzmann in the ramp. The point-particle arm lies on that analytic
curve to 1.2 % area-weighted, which validates the drift kernel at production scale with nothing
fitted. The profile it reproduces demands a central areal packing fraction of **1.37**, against
0.82 for random close packing of disks. The point-particle description is not inaccurate there;
it is reporting a state a monolayer of these molecules cannot occupy. Exclusion caps the centre
at 0.39 and pushes the excess outward.

**Reactive, their full model.** Exclusion suppresses the central density 1.79-fold and cuts core
occupancy by a fifth. Both arms stay below half the close-packing ceiling, so nothing here is
unphysical: exclusion is a real but moderate effect in the reactive arm.

**A negative result worth keeping.** Exclusion changes the plateau, not the approach. Both arms
rise together to about 50 s with the same half-time. At this density, on single-size complexes,
exclusion does not convert their seconds into the physiological tens of minutes. Their hypothesis
was about microclusters, which are larger objects than anything modelled here, so this tests only
the part of it a single-size model can test.

See `NOTES.md` for the six engine traps, the full provenance and the arms that were run and set
aside (the Dharan-Farago force, the pinning obstacles, the neutral gauge).